# Evaluación del NPP-LSTM original


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from utils import (
    datetime_to_hours,
    truncate_catalog_by_threshold,
    append_burn_in_to_test_set
)
from npp_model import construir_npp
from npp_training import preparar_ventanas, perdida_npp

catalogo = pd.read_csv(
    raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"
)

catalogo = datetime_to_hours(catalogo).dropna()
catalogo = truncate_catalog_by_threshold(catalogo, 3.5)

tiempos = catalogo["time"].to_numpy()
magnitudes = catalogo["mw"].to_numpy()

T_train = tiempos[tiempos < 450000]
M_train = magnitudes[tiempos < 450000]

T_test = tiempos[tiempos >= 450000]
M_test = magnitudes[tiempos >= 450000]

# Incorporar el historial anterior, como en el original
T_eval, M_eval = append_burn_in_to_test_set(
    T_train, T_test, M_train, M_test, 20
)

X_test = preparar_ventanas(
    T_eval, M_eval, ventana=20, m0pred=4.5
)

print("Eventos originales de prueba:", len(T_test))
print("Secuencias evaluables:", len(X_test[0]))
print("Eventos objetivo Mw >= 4.5:",
      int(X_test[4].sum()))

Eventos originales de prueba: 820
Secuencias evaluables: 818
Eventos objetivo Mw >= 4.5: 432


In [3]:
mu_t = np.log(np.diff(T_train)).mean()
sigma_t = np.log(np.diff(T_train)).std()
mu_m = np.log(M_train).mean()
sigma_m = np.log(M_train).std()

modelo = construir_npp(
    ventana=20,
    mu_t=mu_t,
    sigma_t=sigma_t,
    mu_m=mu_m,
    sigma_m=sigma_m,
    m0pred=4.5
)

modelo.load_weights(
    str(raiz / "models/NPP_original_W20.weights.h5")
)

print("Pesos recuperados correctamente.")

Pesos recuperados correctamente.


In [4]:
perdida_total = 0.0
intensidades = []
densidades = []

BATCH = 128

for inicio in range(0, len(X_test[0]), BATCH):
    fin = inicio + BATCH

    lote = tuple(
        tf.convert_to_tensor(a[inicio:fin])
        for a in X_test
    )

    loss, intensidad, densidad = perdida_npp(
        modelo, lote, entrenando=False
    )

    perdida_total += float(loss.numpy())
    intensidades.append(intensidad.numpy())
    densidades.append(densidad.numpy())

intensidades = np.concatenate(intensidades).ravel()
densidades = np.concatenate(densidades).ravel()
mascara = X_test[4].ravel().astype(bool)

print("Secuencias evaluadas:", len(X_test[0]))
print("Pérdida media en prueba:",
      perdida_total / len(X_test[0]))

print("Eventos objetivo evaluados:", mascara.sum())

print("Objetivos con densidad casi nula:",
      np.sum(densidades[mascara] <= 1e-10))

print("Intensidades finitas:",
      np.isfinite(intensidades).all())

print("Densidades finitas:",
      np.isfinite(densidades).all())

Secuencias evaluadas: 818
Pérdida media en prueba: 6.9566311777949625
Eventos objetivo evaluados: 432
Objetivos con densidad casi nula: 117
Intensidades finitas: True
Densidades finitas: True


In [5]:
magnitudes_objetivo = X_test[3].ravel()[mascara]
densidades_objetivo = densidades[mascara]

en_umbral = np.isclose(
    magnitudes_objetivo, 4.5, atol=1e-6
)

densidad_cero = densidades_objetivo <= 1e-10

print("Objetivos con Mw = 4.5:", en_umbral.sum())
print("Densidad casi nula:", densidad_cero.sum())

print(
    "Densidad casi nula y Mw = 4.5:",
    np.sum(densidad_cero & en_umbral)
)

print(
    "Densidad casi nula con Mw > 4.5:",
    np.sum(densidad_cero & ~en_umbral)
)

Objetivos con Mw = 4.5: 117
Densidad casi nula: 117
Densidad casi nula y Mw = 4.5: 117
Densidad casi nula con Mw > 4.5: 0
